This is a workflow to find starting points for ATS 2D transect with installed site lat&lon

Determine lat and lon of the starting and ending points of 2D transect

Input information
- lat and lon of the installed sites from field team
- hydrography; NHD plus; HUC and river networks
- DEM
- `config.json`

Output
- `./data/dem/reprojected_dem.tif`
- m2_mat_filename = `../data-processed/{site_name}/startendcoords_{site_name}.mat`
    - `start_coords` and `end_coords`
- it's a perpendicular line representing a hillslope which ends at a specified installed site


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings('ignore', module='pyproj')

# Parameters and data sources

In [ ]:
# Parameters cell
import json
with open('config.json', 'r') as f:
    config = json.load(f)
watershed_name = config['watershed_name']
hucs           = [config['hucs']]
site_name      = config['site_name']

meshsize_nx = config['meshsize_nx']

In [ ]:
def get_huc12(hucs):
    huc12_list = []
    for huc in hucs:
        if len(huc) == 12:
            huc12_list.append(huc)
        elif len(huc) == 10:
            for i in range(1,20):
                huc12_list.append(huc+str(i).zfill(2))
        elif len(huc) == 8:
            for i in range(1,20):
                for j in range(1,20):
                    huc12_list.append(huc+str(i).zfill(2)+str(j).zfill(2))
        else:
            print('need huc8, huc10 or huc12')
    return huc12_list

hucs = get_huc12(hucs)
print(hucs[:10])

In [ ]:
# Parameters cell -- this provides all parameters that can be changed via pipelining to generate a new watershed.
huc_level = 12 # if provided, an int setting the level at which to include HUC boundaries

# geometric parameters
# simplify_hucs = 80 # length scale to target average edge
# simplify_rivers = 30
# stream_outlet_width = 500 # half-width to track a labeled set on which to get discharge
ignore_small_rivers = 2 #default=2 # ignore rivers which have this or fewer reaches.  likely they are irrigation ditches
                        # or other small features which make things complicated but likely don't add much value
prune_by_area_fraction = 0.0 #default=0.01 # ignore reaches whose accumulated catchment area is less than this fraction of the
                              # full domain's area
prune_by_area_fraction_waterbodies = None
# num_smoothing_sweeps = 2 # number of times to smooth the DEM prior to elevating

# # triangle refinement control
include_rivers = True
# # refine_d0 = 100
# # refine_d1 = 500
# # refine_A0 = 8000
# # refine_A1 = 50000
# meshsize = 100
# factor = 5
# refine_d0 = meshsize*3
# refine_A0 = meshsize**2/2
# refine_d1 = meshsize*15
# refine_A1 = (np.round(meshsize*factor))**2/2

# logistics
generate_plots = True # plots take time to make and aren't always needed

In [ ]:
# conda package imports
import os,sys
import numpy as np
import shapely
import pyproj
import itertools
import rasterio
import rasterio.transform
import rasterio.features
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.tri import Triangulation

# Watershed Workflow
import watershed_workflow
import watershed_workflow.source_list
import watershed_workflow.ui
import watershed_workflow.colors
import watershed_workflow.mesh
import watershed_workflow.split_hucs
import watershed_workflow.utils

In [ ]:
# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs = watershed_workflow.crs.daymet_crs()
crs

In [ ]:
# set up a dictionary of source objects
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']
sources['DEM'] = watershed_workflow.source_list.dem_sources['NED 1/3 arc-second']
#sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('/global/cfs/cdirs/m1800/zhi/ww/scripts/data/soil_structure/GLHYMPS/GLHYMPS.shp')
#sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('/global/cfs/cdirs/m1800/zhi/ww/scripts/data/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
#sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('/global/cfs/cdirs/m1800/xiaoyi/ARW-ELMPF/data/ww_data_from_zhi/soil_structure/GLHYMPS/GLHYMPS.shp')
#sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('/global/cfs/cdirs/m1800/xiaoyi/ARW-ELMPF/data/ww_data_from_zhi/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
watershed_workflow.source_list.log_sources(sources)

In [ ]:
# Prepare '../data-processed' folders
os.makedirs(f'../data-processed/{watershed_name}', exist_ok=True)
os.makedirs(f'../data-processed/{site_name}', exist_ok=True)

# Prepare './images' folders
os.makedirs(f'./images/{site_name}', exist_ok=True)
os.makedirs(f'./images/{watershed_name}', exist_ok=True)

# Load watershed DEM

In [ ]:
# load the huc
my_hucs = []
for huc in hucs:
    _, ws = watershed_workflow.get_hucs(sources['HUC'], huc, huc_level, crs)
    my_hucs.extend(ws)

watershed = watershed_workflow.split_hucs.SplitHUCs(my_hucs)

In [ ]:
if include_rivers:  
    # download/collect the river network within that shape's bounds
    _, reaches = watershed_workflow.get_reaches(sources['hydrography'], huc, 
                                                watershed.exterior(), crs, crs,
                                                in_network=True, properties=True)
    
    rivers = watershed_workflow.construct_rivers(reaches, method='hydroseq',
                                                 ignore_small_rivers=ignore_small_rivers,
                                                 prune_by_area=prune_by_area_fraction * watershed.exterior().area * 1.e-6,
                                                 remove_diversions=True,
                                                 remove_braided_divergences=True)
else:
    reaches = []
    rivers = []

In [ ]:
# identify outlets by the elevation map
#watershed_workflow.split_hucs.find_outlets_by_elevation(watershed, crs, dem_sm, dem_profile)

print(len(rivers))
rivers = sorted(rivers, key=len)
#watershed_workflow.split_hucs.find_outlets_by_hydroseq(watershed, rivers[-1])

if generate_plots:
    fig, ax = watershed_workflow.plot.get_ax(crs, figsize=(12,10))
    colors = watershed_workflow.colors.enumerated_colors(len(watershed), palette=4)
    #watershed_workflow.plot.hucs(watershed, crs, ax=ax, color=colors, linewidth=0, facecolor='color', alpha=0.4)
    watershed_workflow.plot.hucs(watershed, crs, ax=ax, color=colors, linewidth=0, facecolor='color', alpha=0.4)
    watershed_workflow.plot.rivers(rivers, crs, ax=ax, colors='b', linewidth=0.5)
    #watershed_workflow.plot.shplys(watershed.polygon_outlets, crs, ax=ax, color=colors, marker='o', markersize=200)

pyproj lat lon to xy
- generate shapely.geometry.point.Point object for each site; for plot later

```
Site_ID	Latitude	Longitude
NF01	46.742847	-120.92965
M01	46.72375	-120.81516
M02	46.729449	-120.9368
M03	46.716332	-121.00993
SF01	46.728668	-120.93757
```

In [ ]:
# lon lat to x y
wgs84 = pyproj.Proj(proj='latlong', datum='WGS84')
proj_daymet = pyproj.Proj('+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs')
lat1, lon1 = 46.742847, -120.92965
lat2, lon2 = 46.72375 , -120.81516
lat3, lon3 = 46.729449, -120.9368
lat4, lon4 = 46.716332, -121.00993
lat5, lon5 = 46.728668, -120.93757

x1, y1 = pyproj.transform(wgs84, proj_daymet, lon1, lat1)
x2, y2 = pyproj.transform(wgs84, proj_daymet, lon2, lat2)
x3, y3 = pyproj.transform(wgs84, proj_daymet, lon3, lat3)
x4, y4 = pyproj.transform(wgs84, proj_daymet, lon4, lat4)
x5, y5 = pyproj.transform(wgs84, proj_daymet, lon5, lat5)

print(f"Longitude: {lon1}, Latitude: {lat1}")
print(f"X: {x1}, Y: {y1}")
print(f"Longitude: {lon2}, Latitude: {lat2}")
print(f"X: {x2}, Y: {y2}")
print(f"Longitude: {lon3}, Latitude: {lat3}")
print(f"X: {x3}, Y: {y3}")
print(f"Longitude: {lon4}, Latitude: {lat4}")
print(f"X: {x4}, Y: {y4}")


In [ ]:
# x y to shapely Point, and plot
site1_xy = shapely.geometry.Point(x1, y1)
site2_xy = shapely.geometry.Point(x2, y2)
site3_xy = shapely.geometry.Point(x3, y3)
site4_xy = shapely.geometry.Point(x4, y4)
site5_xy = shapely.geometry.Point(x5, y5)

fig, ax = watershed_workflow.plot.get_ax(crs, figsize=(9,7.5))
#colors = watershed_workflow.colors.enumerated_colors(len(watershed), palette=4)
#colors = watershed_workflow.colors.enumerated_colors(1, palette=4)
watershed_workflow.plot.hucs(watershed, crs, ax=ax, color=colors[0], linewidth=1, facecolor='color', alpha=0.4)
watershed_workflow.plot.rivers(rivers, crs, ax=ax, colors='b', linewidth=0.5)

watershed_workflow.plot.shplys(site1_xy, crs, ax=ax, color=colors[1], marker='o', markersize=100)
ax.text(x1, y1, 'NF01', fontsize=10, color='k', ha='right', va='bottom', transform=ax.transData)
watershed_workflow.plot.shplys(site2_xy, crs, ax=ax, color=colors[1], marker='o', markersize=100)
ax.text(x2, y2, 'M01', fontsize=10, color='k', ha='right', va='bottom', transform=ax.transData)
watershed_workflow.plot.shplys(site3_xy, crs, ax=ax, color=colors[1], marker='o', markersize=100)
ax.text(x3, y3, 'M02', fontsize=10, color='k', ha='left', va='top', transform=ax.transData)
watershed_workflow.plot.shplys(site4_xy, crs, ax=ax, color=colors[1], marker='o', markersize=100)
ax.text(x4, y4, 'M03', fontsize=10, color='k', ha='right', va='bottom', transform=ax.transData)
watershed_workflow.plot.shplys(site5_xy, crs, ax=ax, color=colors[1], marker='o', markersize=100)
ax.text(x5, y5, 'SF01', fontsize=10, color='k', ha='right', va='bottom', transform=ax.transData)

# remove frame
ax.set_frame_on(False)

# if generate_plots:
#     output_filename = './images/fig-watershedsites.tif'
#     fig.savefig(output_filename, dpi=300, pil_kwargs={'compression': 'tiff_lzw'})

In [ ]:
# # about what is river.preOrder and how to get xy from river segment
# fig, ax = watershed_workflow.plot.get_ax(crs, figsize=(12,10))
# colors = watershed_workflow.colors.enumerated_colors(len(watershed), palette=4)
# watershed_workflow.plot.hucs(watershed, crs, ax=ax, color=colors, linewidth=1, facecolor='color', alpha=0.4)
# watershed_workflow.plot.rivers(rivers, crs, ax=ax, colors='b', linewidth=0.5)

# for river in rivers:
#     #for node in river.preOrder() 
#     for node in itertools.islice(river.preOrder(),100):
#         x,y=node.segment.xy 
#         ax.plot(x,y,'-o',markersize=5)


# DEM pre-process

In [ ]:
# download the needed rasters
dem_profile, dem = watershed_workflow.get_raster_on_shape(sources['DEM'], watershed.exterior(), crs)

if generate_plots:
    fig, axs = plt.subplots(1,1, figsize=(5,5))
    im1 = axs.imshow(dem, cmap='terrain')
    axs.set_title('unsmoothed DEM')
    fig.colorbar(im1, ax=axs, orientation='horizontal', pad=0.1, label='Elevation (Z)')
    

In [ ]:
dem_profile

In [ ]:
## play watershed_workflow code here, testing xy conversion
# import rasterio
# import rasterio.transform
# import rasterio.features

# # from values_from_raster() in __init__.py
# points = np.array([[x1,y1], [x2,y2], [x3,y3]])
# points_crs = crs
# raster = dem
# raster_profile = dem_profile

# raster_crs = watershed_workflow.crs.from_rasterio(raster_profile['crs'])
# points_raster_crs = np.array(
#     watershed_workflow.warp.xy(points[:, 0], points[:, 1], points_crs, raster_crs)).transpose()
# out = raster[rasterio.transform.rowcol(raster_profile['transform'], points_raster_crs[:, 0],
#                                         points_raster_crs[:, 1])]

# print(points)
# print(points_raster_crs)
# print(rasterio.transform.rowcol(raster_profile['transform'], points_raster_crs[:, 0],
#                                         points_raster_crs[:, 1]))

## M2: convert raster crs first, then construct mesh from raster
M1 has been deprecated: use watershed_workflow.warp.xy to obtain mesh under daymet crs

In [ ]:
# download the needed rasters
dem_profile2, dem2 = watershed_workflow.get_raster_on_shape(sources['DEM'], watershed.exterior(), crs, out_crs=crs)

if generate_plots:
    # Calculate extent from the transform for geographic coordinates
    transform = dem_profile2['transform']
    rows, cols = dem2.shape
    left = transform.c
    right = transform.c + cols * transform.a
    top = transform.f
    bottom = transform.f + rows * transform.e
    extent = [left, right, bottom, top]
    
    fig, ax = plt.subplots(1,1, figsize=(5,5))
    im1 = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper')
    ax.set_title('DEM under Daymet crs')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    fig.colorbar(im1, ax=ax, orientation='horizontal', pad=0.1, label='Elevation (Z)')

In [ ]:
print(crs)
print(dem_profile2['crs'])

In [ ]:
output_path = "./data/dem/reprojected_dem.tif"
with rasterio.open(output_path, 'w', **dem_profile2) as dst:
    dst.write(dem2, 1)

In [ ]:
if generate_plots:
    # Calculate extent from the transform for geographic coordinates
    transform = dem_profile2['transform']
    rows, cols = dem2.shape
    left = transform.c
    right = transform.c + cols * transform.a
    top = transform.f
    bottom = transform.f + rows * transform.e
    extent = [left, right, bottom, top]
    
    fig, ax = plt.subplots(1,1, figsize=(10,10))
    im1 = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper')
    #ax.set_title('DEM under Daymet CRS')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    fig.colorbar(im1, ax=ax, orientation='horizontal', pad=0.1, label='Elevation (Z)')

    watershed_workflow.plot.hucs(watershed, crs, ax=ax, color='k', linewidth=0.75)
    watershed_workflow.plot.rivers(rivers, crs, ax=ax, color='red', linewidth=0.5)
    
    watershed_workflow.plot.shplys(site1_xy, crs, ax=ax, color='blue', marker='o', markersize=100)
    watershed_workflow.plot.shplys(site2_xy, crs, ax=ax, color='blue', marker='o', markersize=100)
    watershed_workflow.plot.shplys(site3_xy, crs, ax=ax, color='blue', marker='o', markersize=100)
    watershed_workflow.plot.shplys(site4_xy, crs, ax=ax, color='blue', marker='o', markersize=100)
    watershed_workflow.plot.shplys(site5_xy, crs, ax=ax, color='blue', marker='o', markersize=100)

In [ ]:
# zoom in to one site - site NF01 e.g.
if generate_plots:
    # site NF01
    dx = 5000/2
    dy = 4000/2
    xmin = x1 - dx/2
    xmax = x1 + dx/2
    ymin = y1 - dy/2
    ymax = y1 + dy/2
    
    transform = dem_profile2['transform']
    rows, cols = dem2.shape
    left = transform.c
    right = transform.c + cols * transform.a
    top = transform.f
    bottom = transform.f + rows * transform.e
    extent = [left, right, bottom, top]
    fig, ax = plt.subplots(1,1, figsize=(7.5,6))
    im1 = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper')
    #ax.set_title('DEM under Daymet CRS')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    fig.colorbar(im1, ax=ax, orientation='horizontal', pad=0.1, label='Elevation (Z)')

    watershed_workflow.plot.hucs(watershed, crs, ax=ax, color='k', linewidth=1)
    watershed_workflow.plot.rivers(rivers, crs, ax=ax, color='red', linewidth=1)
    
    watershed_workflow.plot.shplys(site1_xy, crs, ax=ax, color='blue', marker='o', markersize=100)
    watershed_workflow.plot.shplys(site2_xy, crs, ax=ax, color='blue', marker='o', markersize=100)
    watershed_workflow.plot.shplys(site3_xy, crs, ax=ax, color='blue', marker='o', markersize=100)
    watershed_workflow.plot.shplys(site4_xy, crs, ax=ax, color='blue', marker='o', markersize=100)
    watershed_workflow.plot.shplys(site5_xy, crs, ax=ax, color='blue', marker='o', markersize=100)
    
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

# Find 2D transect start/end points

input:
- site lat lon
- river network
- 3D mesh

In [ ]:
# 1. find the closest point in the "rivers"
def point_to_segment_distance(x1, y1, x2, y2, px, py):
    """
    Calculate the minimum distance between a point (px, py)
    and a line segment defined by (x1, y1) and (x2, y2).
    """
    dx, dy = x2 - x1, y2 - y1
    if dx == 0 and dy == 0:  # The segment is a single point
        return np.hypot(px - x1, py - y1)

    # Project point onto the line segment
    t = max(0, min(1, ((px - x1) * dx + (py - y1) * dy) / (dx * dx + dy * dy)))
    proj_x, proj_y = x1 + t * dx, y1 + t * dy

    # Return the Euclidean distance
    return np.hypot(px - proj_x, py - proj_y)

def find_closest_segment_with_index(rivers, point):
    x1, y1 = point
    min_dist = float('inf')
    closest_segment = None
    closest_node_index = None
    closest_node = None

    for river_idx, river in enumerate(rivers):
        for node_idx, node in enumerate(itertools.islice(river.preOrder(), 0, None)):
            x, y = node.segment.xy
            for i in range(len(x) - 1):  # Iterate through segments
                dist = point_to_segment_distance(x[i], y[i], x[i+1], y[i+1], x1, y1)
                if dist < min_dist:
                    min_dist = dist
                    closest_segment = ((x[i], y[i]), (x[i+1], y[i+1]))
                    closest_node_index = (river_idx, node_idx)
                    closest_node = node

    return closest_segment, min_dist, closest_node_index, closest_node

# Example usage
point = [x1, y1]
closest_segment, min_distance, node_index, node = find_closest_segment_with_index(rivers, point)

print(f"Closest segment: {closest_segment}")
print(f"Minimum distance: {min_distance}")
print(f"Closest node index (river, node): {node_index}")
print(f"Closest node: {node}")

# Plot the river network and the closest segment
# import matplotlib.pyplot as plt

fig, ax = watershed_workflow.plot.get_ax(crs, figsize=(10,8))
colors = watershed_workflow.colors.enumerated_colors(len(watershed), palette=4)
watershed_workflow.plot.hucs(watershed, crs, ax=ax, color=colors[0], linewidth=1, facecolor='color', alpha=0.4)
watershed_workflow.plot.rivers(rivers, crs, ax=ax, colors='b', linewidth=0.5)

# # Highlight the closest segment
# x_seg, y_seg = zip(*closest_segment)
# ax.plot(x_seg, y_seg, '-o', color='red', markersize=8, label='Closest Segment')

river=rivers[node_index[0]]
node = next(itertools.islice(river.preOrder(), node_index[1], node_index[1] + 1))
x,y=node.segment.xy 
ax.plot(x,y,'-o', color='red', markersize=5, label='Closest river node')

# Highlight the specified point
ax.plot(x1, y1, 'x', color='blue', markersize=10, label='Specified Point')

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

plt.legend()
plt.show()

In [ ]:
# 2. find the direction of river by smoothing the river segment, and obtain the perpendicular direction

#import numpy as np
from scipy.ndimage import gaussian_filter1d
#import matplotlib.pyplot as plt

def smooth_segment_gaussian(x, y, sigma=2):
    """
    Smooth the segment using a Gaussian filter.
    Args:
        x, y: Coordinates of the segment.
        sigma: Standard deviation for Gaussian kernel.
    Returns:
        Smoothed x and y coordinates.
    """
    x_smooth = gaussian_filter1d(x, sigma)
    y_smooth = gaussian_filter1d(y, sigma)
    return x_smooth, y_smooth

def get_direction_perpendicular_to_segment(x, y, point):
    """
    Compute the direction perpendicular to the smoothed segment at the closest point.
    Args:
        x, y: Coordinates of the (smoothed) segment.
        point: The specified point [px, py].
    Returns:
        Perpendicular direction vector at the closest point.
    """
    px, py = point

    # Find the closest point on the smoothed curve
    distances = np.hypot(x - px, y - py)
    closest_idx = np.argmin(distances)

    # Compute the tangent vector at the closest point
    if closest_idx == 0:  # First point
        dx, dy = x[1] - x[0], y[1] - y[0]
    elif closest_idx == len(x) - 1:  # Last point
        dx, dy = x[-1] - x[-2], y[-1] - y[-2]
    else:  # Middle points
        dx = x[closest_idx + 1] - x[closest_idx - 1]
        dy = y[closest_idx + 1] - y[closest_idx - 1]

    # Normalize the tangent vector
    tangent = np.array([dx, dy])
    tangent /= np.linalg.norm(tangent)

    # Compute the perpendicular vector (rotate by 90 degrees)
    perpendicular = np.array([-tangent[1], tangent[0]])

    return perpendicular, (x[closest_idx], y[closest_idx])

# Example usage
#x = np.linspace(0, 10, 100)
#y = np.sin(x)  # Example curve
#point = [5, 0]  # Specified point
river=rivers[node_index[0]]
node = next(itertools.islice(river.preOrder(), node_index[1], node_index[1] + 1))
x,y=node.segment.xy 
point = [x1, y1]

# Smooth the curve using Gaussian filter
x_smooth, y_smooth = smooth_segment_gaussian(x, y, sigma=2)

# Get the perpendicular direction
perpendicular, closest_point = get_direction_perpendicular_to_segment(x_smooth, y_smooth, point)

# Plot
fig, ax = watershed_workflow.plot.get_ax(crs, figsize=(10,8))
colors = watershed_workflow.colors.enumerated_colors(len(watershed), palette=4)
watershed_workflow.plot.hucs(watershed, crs, ax=ax, color=colors[0], linewidth=1, facecolor='color', alpha=0.4)
watershed_workflow.plot.rivers(rivers, crs, ax=ax, colors='b', linewidth=0.5)

river=rivers[node_index[0]]
node = next(itertools.islice(river.preOrder(), node_index[1], node_index[1] + 1))
x,y=node.segment.xy 
ax.plot(x,y,'--', color='red', markersize=5, label='Original Curve', alpha=0.5)
ax.plot(x_smooth, y_smooth,'-o', color='green', markersize=5, label='Smoothed Curve', alpha=0.5)
ax.plot(*point, 'bx', label='Specified Point')
ax.plot(*closest_point, 'gx', label='Closest Point on Curve')
ax.quiver(*closest_point, *perpendicular, angles='xy', scale_units='xy', scale=3e-3, color='purple', label='Perpendicular Direction')
ax.quiver(*closest_point, *(-perpendicular), angles='xy', scale_units='xy', scale=3e-3, color='orange', label='Reverse Perpendicular Direction')

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

plt.legend()


In [ ]:
# 3. searching along the perpendicular direction with DEM info, and obtain the starting point of hillslope model
delta_distance_xy = 10 # searching distance at xy plane, unit meter
maximum_distance= [1000,1500] # maximum searching distance along perpendicular and -perpendicular direction

## perpendicular dir
distances_posdir = np.arange(0, maximum_distance[0] + delta_distance_xy, delta_distance_xy)
points_posdir = np.array([closest_point + d * perpendicular for d in distances_posdir])
rows, cols = rasterio.transform.rowcol(dem_profile2['transform'], points_posdir[:,0], points_posdir[:,1])
rows_pos = np.array(rows)
cols_pos = np.array(cols)
points_z_posdir = dem2[rows_pos, cols_pos]

## -perpendicular dir
distances_negdir = np.arange(0, maximum_distance[1] + delta_distance_xy, delta_distance_xy)
points_negdir = np.array([closest_point + d * (-perpendicular) for d in distances_negdir])
rows, cols = rasterio.transform.rowcol(dem_profile2['transform'], points_negdir[:,0], points_negdir[:,1])
rows_neg = np.array(rows)
cols_neg = np.array(cols)
points_z_negdir = dem2[rows_neg, cols_neg]

# Plot
fig, ax = watershed_workflow.plot.get_ax(crs, figsize=(10,8))
colors = watershed_workflow.colors.enumerated_colors(len(watershed), palette=4)
watershed_workflow.plot.hucs(watershed, crs, ax=ax, color=colors[0], linewidth=1, facecolor='color', alpha=0.4)
watershed_workflow.plot.rivers(rivers, crs, ax=ax, colors='b', linewidth=0.5)

river=rivers[node_index[0]]
node = next(itertools.islice(river.preOrder(), node_index[1], node_index[1] + 1))
x,y=node.segment.xy 
ax.plot(x,y,'--', color='red', markersize=5, label='Original Curve', alpha=0.5)
ax.plot(x_smooth, y_smooth,'-o', color='green', markersize=5, label='Smoothed Curve', alpha=0.5)
ax.plot(*point, 'bx', label='Specified Point')
ax.plot(*closest_point, 'gx', label='Closest Point on Curve')
ax.quiver(*closest_point, *perpendicular, angles='xy', scale_units='xy', scale=3e-3, color='purple', label='Perpendicular Direction')
ax.quiver(*closest_point, *(-perpendicular), angles='xy', scale_units='xy', scale=3e-3, color='orange', label='Reverse Perpendicular Direction')
ax.plot(points_posdir[:,0],points_posdir[:,1],'-o', color='purple',markersize=5, alpha=0.5)
ax.plot(points_negdir[:,0],points_negdir[:,1],'-o', color='orange',markersize=5, alpha=0.5)

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

plt.legend()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(10,6))

ax.plot(distances_posdir,points_z_posdir, '--', color = 'purple')
ax.plot((-distances_negdir),points_z_negdir, '--', color = 'orange')


## Determine the end point of 2D transect

In [ ]:
from scipy.signal import find_peaks, savgol_filter

smoothed_z_posdir = savgol_filter(points_z_posdir, window_length=21, polyorder=3)
peaks_posdir, properties = find_peaks(smoothed_z_posdir, prominence=1)
smoothed_z_negdir = savgol_filter(points_z_negdir, window_length=21, polyorder=3)
peaks_negdir, properties = find_peaks(smoothed_z_negdir, prominence=1)

fig, ax = plt.subplots(1,1, figsize=(10,6))

ax.plot(distances_posdir,points_z_posdir, '--', label="Original Line", color = "purple", linewidth = 2, alpha = 0.6)
ax.plot(distances_posdir, smoothed_z_posdir, label="Smoothed Line", color="purple", linewidth = 1, alpha = 0.8)
ax.plot(-distances_negdir,points_z_negdir, '--', label="Original Line", color = 'orange', linewidth = 2, alpha = 0.6)
ax.plot(-distances_negdir, smoothed_z_negdir, label="Smoothed Line", color="orange", linewidth = 1, alpha = 0.8)

ax.scatter(distances_posdir[peaks_posdir], smoothed_z_posdir[peaks_posdir], color="red", label="Peaks", zorder = 5)
ax.scatter(-distances_negdir[peaks_negdir], smoothed_z_negdir[peaks_negdir], color="red", label="Peaks", zorder = 5)


In [ ]:
print(peaks_posdir)
print(peaks_negdir)

In [ ]:
select_peaks_posdir_index = 0
select_peaks_negdir_index = 0
print("along perpendicular direction")
print("peak x:", points_posdir[peaks_posdir[select_peaks_posdir_index],0])
print("peak y:", points_posdir[peaks_posdir[select_peaks_posdir_index],1])
print("peak z:", points_z_posdir[peaks_posdir[select_peaks_posdir_index]])

print("along -perpendicular direction")
print("peak x:", points_negdir[peaks_negdir[select_peaks_negdir_index],0])
print("peak y:", points_negdir[peaks_negdir[select_peaks_negdir_index],1])
print("peak z:", points_z_negdir[peaks_negdir[select_peaks_negdir_index]])

print("river site point")
print("site x:", points_posdir[0,0])
print("site y:", points_posdir[0,1])
print("site z:", points_z_posdir[0])

## plot 2D transect in 3D

In [ ]:
# # Get the shape of the DEM
# rows, cols = dem2.shape

# # Generate the row and column indices
# row_indices, col_indices = np.meshgrid(np.arange(rows), np.arange(cols), indexing="ij")

# # Use rasterio's transform to compute geographic coordinates (x, y)
# xs, ys = rasterio.transform.xy(dem_profile2['transform'], row_indices, col_indices, offset="center")
# xs = np.array(xs)
# ys = np.array(ys)

# # Flatten the arrays for easier processing
# xs_raster_crs = xs.flatten()
# ys_raster_crs = ys.flatten()
# zs_raster_crs = dem2.flatten()

# # Filter out points where the DEM has NaN values
# valid_indices = ~np.isnan(zs_raster_crs)
# xs_raster_crs = xs_raster_crs[valid_indices]
# ys_raster_crs = ys_raster_crs[valid_indices]
# zs_raster_crs = zs_raster_crs[valid_indices]

# Use watershed_workflow.warp.xy to compute points_crs
raster_crs = watershed_workflow.crs.from_rasterio(dem_profile2['crs'])
# points_crs = np.array(
#     watershed_workflow.warp.xy(xs_raster_crs, ys_raster_crs, raster_crs, crs))

In [ ]:
row_min = np.min(np.concatenate((rows_pos,rows_neg)))
row_max = np.max(np.concatenate((rows_pos,rows_neg)))
col_min = np.min(np.concatenate((cols_pos,cols_neg)))
col_max = np.max(np.concatenate((cols_pos,cols_neg)))

row0 = rows_pos[0]
col0 = cols_pos[0]

ratio_set = 5/4

if (col_max-col_min)/(row_max-row_min) > ratio_set:
    #increase row_max-row_min
    col_max_crop = col_max
    col_min_crop = col_min
    row_max_crop = row0 + np.floor((row_max-row0)*(col_max-col_min)/5*4/(row_max-row_min)).astype(int)
    row_min_crop = row0 - np.floor((row0-row_min)*(col_max-col_min)/5*4/(row_max-row_min)).astype(int)
else:
    #increase col_max-col_min
    row_max_crop = row_max
    row_min_crop = row_min
    col_max_crop = col0 + np.floor((col_max-col0)*(row_max-row_min)/4*5/(col_max-col_min)).astype(int)
    col_min_crop = col0 - np.floor((col0-col_min)*(row_max-row_min)/4*5/(col_max-col_min)).astype(int)

print("row_min = ", row_min, " row_max = ", row_max)
print("col_min = ", col_min, " col_max = ", col_max)
print("row_min_crop = ", row_min_crop, " row_max_crop = ", row_max_crop)
print("col_min_crop = ", col_min_crop, " col_max_crop = ", col_max_crop)

In [ ]:
## generate mesh within row_min_crop~row_max_crop and col_min_crop~col_max_crop
# Crop the DEM and indices
cropped_dem = dem2[row_min_crop:row_max_crop, col_min_crop:col_max_crop]
cropped_rows, cropped_cols = cropped_dem.shape

# Generate the row and column indices for the cropped DEM
cropped_row_indices, cropped_col_indices = np.meshgrid(
    np.arange(row_min_crop, row_max_crop), 
    np.arange(col_min_crop, col_max_crop), 
    indexing="ij"
)

# Compute geographic coordinates (x, y) for the cropped DEM
cropped_xs, cropped_ys = rasterio.transform.xy(
    dem_profile2['transform'], 
    cropped_row_indices, 
    cropped_col_indices, 
    offset="center"
)
cropped_xs = np.array(cropped_xs)
cropped_ys = np.array(cropped_ys)

# Flatten the arrays for easier processing
cropped_xs_raster_crs = cropped_xs.flatten()
cropped_ys_raster_crs = cropped_ys.flatten()
cropped_zs_raster_crs = cropped_dem.flatten()

# Transform to points_crs
cropped_points_crs = np.array(
    watershed_workflow.warp.xy(
        cropped_xs_raster_crs, 
        cropped_ys_raster_crs, 
        raster_crs, 
        crs
    )
)

# Create a 2D grid of indices for the cropped raster
cropped_grid_indices = np.arange(cropped_rows * cropped_cols).reshape(cropped_rows, cropped_cols)

# Define the triangles for the cropped DEM
cropped_triangles = []
for i in range(cropped_rows - 1):
    for j in range(cropped_cols - 1):
        # Indices of the four corners of a quad
        p1 = cropped_grid_indices[i, j]
        p2 = cropped_grid_indices[i, j + 1]
        p3 = cropped_grid_indices[i + 1, j]
        p4 = cropped_grid_indices[i + 1, j + 1]

        # Two triangles for each quad
        cropped_triangles.append([p1, p2, p3])
        cropped_triangles.append([p2, p4, p3])

cropped_triangles = np.array(cropped_triangles)

# Combine the coordinates into vertices
cropped_vertices_crs = np.column_stack((
    cropped_points_crs[0, :], 
    cropped_points_crs[1, :], 
    cropped_zs_raster_crs
))

In [ ]:
# plot 3D mesh

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

# Create a new figure
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Extract the x, y, and z coordinates for all vertices
x = cropped_vertices_crs[:, 0]
y = cropped_vertices_crs[:, 1]
z = cropped_vertices_crs[:, 2]

# Plot the surface mesh using the triangles and the vertices
ax.plot_trisurf(x, y, z, triangles=cropped_triangles, cmap=cm.viridis, linewidth=0.1, edgecolor="gray", alpha=0.25)

# Add the first scatter plot
ax.scatter(
    points_posdir[:, 0],  # x-coordinates
    points_posdir[:, 1],  # y-coordinates
    points_z_posdir,      # z-coordinates
    color="purple",       # Scatter point color
    label="Positive Direction", 
    s=20,                 # Point size
    depthshade=True,      # Enable depth shading
    alpha=0.5
)

# Add the second scatter plot
ax.scatter(
    points_negdir[:, 0],  # x-coordinates
    points_negdir[:, 1],  # y-coordinates
    points_z_negdir,      # z-coordinates
    color="orange",       # Scatter point color
    label="Negative Direction", 
    s=20,                 # Point size
    depthshade=True,      # Enable depth shading
    alpha=0.5
)

# Add end points of 2D transects
marker_size=200
scatter_marker="X"
ax.scatter(
    points_posdir[0,0],   # x-coordinates
    points_posdir[0,1],   # y-coordinates
    points_z_negdir[0],   # z-coordinates
    color="red",          # Scatter point color
    s=marker_size,        # Point size
    marker=scatter_marker
)
ax.scatter(
    points_posdir[peaks_posdir[select_peaks_posdir_index],0],   # x-coordinates
    points_posdir[peaks_posdir[select_peaks_posdir_index],1],   # y-coordinates
    points_z_posdir[peaks_posdir[select_peaks_posdir_index]],   # z-coordinates
    color="red",          # Scatter point color
    s=marker_size,        # Point size
    marker=scatter_marker
)
ax.scatter(
    points_negdir[peaks_negdir[select_peaks_negdir_index],0],   # x-coordinates
    points_negdir[peaks_negdir[select_peaks_negdir_index],1],   # y-coordinates
    points_z_negdir[peaks_negdir[select_peaks_negdir_index]],   # z-coordinates
    color="red",          # Scatter point color
    s=marker_size,        # Point size
    marker=scatter_marker
)


ax.view_init(elev=30, azim=-50)

# Set axis labels
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

# Set title
ax.set_title('3D Surface Mesh')

ax.set_box_aspect([1,1,0.5])

# Show the plot
plt.show()


In [ ]:
print(points_posdir[peaks_posdir[select_peaks_posdir_index],:])
print(points_negdir[peaks_negdir[select_peaks_negdir_index],:])
print(points_posdir[0,:])

In [ ]:
# start_coords, end_coords = (-1511015.74507677, 640547.2830956738), (-1511677.2648839361, 640389.829786001) # try to change from 440450 to 440850_BL
# start_coords2 = (-1512611.1751999462, 640167.5427606269)
start_coords  = (points_posdir[peaks_posdir[select_peaks_posdir_index],0], points_posdir[peaks_posdir[select_peaks_posdir_index],1])
end_coords    = (points_posdir[0,0], points_posdir[0,1])
start_coords2 = (points_negdir[peaks_posdir[select_peaks_posdir_index],0], points_negdir[peaks_posdir[select_peaks_posdir_index],1])

## save start/end_coords to mat file

In [ ]:
# save start_coords and end_coords, for notebook "1-full_workflow_OakCreek.NF01.ipynb"
## site_name = 'NF01' # in config.json now

from scipy.io import savemat

data_to_save = {
    'start_coords': start_coords,
    'end_coords': end_coords,
    'start_coords2': start_coords2
}

m2_mat_filename =  f'../data-processed/{site_name}/startendcoords_{site_name}.mat'
savemat(m2_mat_filename, data_to_save)

print(f"start end pts coords saved. {m2_mat_filename}")

In [ ]:
# site NF01

dx = 5000/2
dy = 4000/2
xmin = x1 - dx/2
xmax = x1 + dx/2
ymin = y1 - dy/2
ymax = y1 + dy/2

fig, ax = plt.subplots(1,1, figsize=(5*0.75,4*0.75))

# 2D mesh plot
transform = dem_profile2['transform']
rows, cols = dem2.shape
left = transform.c
right = transform.c + cols * transform.a
top = transform.f
bottom = transform.f + rows * transform.e
extent = [left, right, bottom, top]

im1 = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper')
#ax.set_title('DEM under Daymet CRS')
#ax.set_xlabel('Easting (m)')
#ax.set_ylabel('Northing (m)')
#fig.colorbar(im1, ax=ax, orientation='horizontal', pad=0.1, label='Elevation (Z)')
# Remove x and y tick marks and labels
ax.tick_params(axis='both', which='both', length=0, labelbottom=False, labelleft=False)

watershed_workflow.plot.hucs(watershed, crs, ax=ax, color='k', linewidth=1)
watershed_workflow.plot.rivers(rivers, crs, ax=ax, color='red', linewidth=1)

ax.plot([start_coords[0], end_coords[0]], [start_coords[1], end_coords[1]], 'orange', linewidth=3)
ax.plot([start_coords2[0], end_coords[0]], [start_coords2[1], end_coords[1]], 'purple', linewidth=3)
watershed_workflow.plot.shplys(site1_xy, crs, ax=ax, color='blue', marker='x', markersize=100)
ax.plot(start_coords[0], start_coords[1], 'bx', markersize=10)
ax.plot(start_coords2[0], start_coords2[1], 'bx', markersize=10)
mid_x_hill1 = (start_coords[0] + end_coords[0]) / 2
mid_y_hill1 = (start_coords[1] + end_coords[1]) / 2
ax.text(mid_x_hill1, mid_y_hill1, 'Hillslope 1', fontsize=10, color='k', ha='right', va='bottom')
mid_x_hill2 = (start_coords2[0] + end_coords[0]) / 2
mid_y_hill2 = (start_coords2[1] + end_coords[1]) / 2
ax.text(mid_x_hill2, mid_y_hill2, 'Hillslope 2', fontsize=10, color='k', ha='right', va='bottom')

plt.colorbar(im1, ax=ax, label='Elevation (m)')

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# if generate_plots:
#     output_filename = './images/fig-siteNF01-hillslopes.tif'
#     fig.savefig(output_filename, dpi=300, pil_kwargs={'compression': 'tiff_lzw'})

In [ ]:
# Print coordinates and calculate distance
print(f"start_coords: {start_coords}")
print(f"end_coords: {end_coords}")

# Calculate distance between start_coords and end_coords
distance = np.sqrt((start_coords[0] - end_coords[0])**2 + (start_coords[1] - end_coords[1])**2)
print(f"\nDistance between start_coords and end_coords: {distance:.2f} meters")

# Calculate resolution along x axis
resolution = distance / meshsize_nx
print(f"Resolution along x axis (distance/meshsize_nx): {resolution:.2f} meters")